# Projeto integrado. Gestão da base de conhecimento do atendimento da Quantum Commerce

Knowledge Management and Prompt Engineering. MBA AI Engineering & Multi-Agents.

**Integrantes do grupo.** [nomes completos, um por linha]

Este notebook é o esqueleto do trabalho. As seções seguem a ordem do enunciado. As células marcadas com `SUA VEZ` são as células do grupo. As células de análise, em markdown, vêm depois de cada medição e recebem a leitura do número e a decisão que ele motivou. O notebook precisa rodar do início ao fim, sem célula que dependa de ação manual.

---
Material desenvolvido por  
**prof. Thiago Nascimento Nogueira** (profthiago.nogueira@fiap.com.br)  
Adaptado por  
**prof. Alexandre G. L. Fernandes** (profalexandre.fernandes@fiap.com.br)  

## 0. Configuração e dados

In [ ]:
# Instalação. Acrescente aqui o que o grupo usar.
!pip install -q sentence-transformers rank_bm25 anthropic openai ragas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.2/353.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.

In [ ]:
import os
from getpass import getpass

# ========================== SUA VEZ ==========================
# Escolha entre 'Anthropic' e 'OpenAI'
MODEL_PROVIDER = 'OpenAI' # @param ['Anthropic', 'OpenAI']

cliente = None

if MODEL_PROVIDER == 'Anthropic':
    import anthropic
    if 'ANTHROPIC_API_KEY' not in os.environ:
        os.environ['ANTHROPIC_API_KEY'] = getpass('Chave da API do Anthropic: ')
    cliente = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
elif MODEL_PROVIDER == 'OpenAI':
    from openai import OpenAI
    if 'OPENAI_API_KEY' not in os.environ:
        os.environ['OPENAI_API_KEY'] = getpass('Chave da API do OpenAI: ')
    cliente = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
else:
    raise ValueError('MODEL_PROVIDER inválido.')

print(f"Usando o provedor de modelo: {MODEL_PROVIDER}")

Chave da API do OpenAI: ··········
Usando o provedor de modelo: OpenAI


In [ ]:
MAX_TOKENS = 2000

def get_llm_response(messages, max_tokens=MAX_TOKENS):
    """Abstrai a chamada da API do LLM com base no MODEL_PROVIDER escolhido."""
    if MODEL_PROVIDER == 'Anthropic':
        response = cliente.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            messages=messages,
        )
        return response.content[0].text
    elif MODEL_PROVIDER == 'OpenAI':
        response = cliente.chat.completions.create(
            # model="gpt-5.4-mini",
            model="gpt-5.6-luna", # testando esse modelo
            messages=messages,
            max_completion_tokens=max_tokens,
        )
        return response.choices[0].message.content

In [ ]:
get_llm_response([{"role": "user", "content": "Olá! Qual é o seu provedor?"}])

'Olá! Sou um modelo de linguagem desenvolvido pela OpenAI.'

In [ ]:
# Os quatro conjuntos de dados, carregados por link.
import urllib.request, zipfile, io, json, csv, os

BASE = "https://raw.githubusercontent.com/thiagonogueira/datasets/main/quantum-commerce/"
#BASE = "https://raw.githubusercontent.com/aglfern/fiap-dataset-km2026/main/"

with urllib.request.urlopen(BASE + "central-documentos.zip") as r:
    zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")
manifest = json.load(open("central-documentos/manifest.json", encoding="utf-8"))
DOCS = {k: open(f"central-documentos/{k}.md", encoding="utf-8").read() for k in manifest}

def baixa_csv(nome):
    with urllib.request.urlopen(BASE + nome) as r:
        return list(csv.DictReader(io.StringIO(r.read().decode("utf-8")), delimiter=";"))

chamados = baixa_csv("chamados-200.csv")
perguntas = baixa_csv("perguntas-referencia.csv")

print(len(DOCS), "documentos.", len(chamados), "chamados.", len(perguntas), "perguntas de referência.")

44 documentos. 200 chamados. 62 perguntas de referência.


In [ ]:
# Um documento, como exemplo. Título, público, estrutura e vigência vêm do manifesto.
k = "pol-dev-03"
print(manifest[k])
print(DOCS[k][:800])

{'titulo': 'Política de trocas, devoluções e ressarcimento (POL DEV 03, versão 3)', 'publico': 'cliente', 'estrutura': 'prosa_tabela', 'vigencia': '2026-07-01/9999-12-31'}
# Política de Trocas, Devoluções e Ressarcimento (POL DEV 03, versão 3)

Vigente a partir de 01/07/2026. Substitui a versão 2.

---

## Objetivo e Âmbito

Esta política estabelece as regras aplicáveis a todos os pedidos de troca, devolução e ressarcimento realizados por meio da plataforma. O documento é válido para compras efetuadas por pessoas físicas e jurídicas cadastradas como compradores ativos, independentemente da categoria de produto adquirida, do valor da compra ou da modalidade de pagamento utilizada. As regras aqui descritas se aplicam a todas as categorias de produtos disponíveis na plataforma, salvo disposição expressa em contrário indicada em política específica de categoria.

A plataforma atua como intermediadora entre comprador e vendedor e garante o cumprimento destas regr


## 1. Linha de base

O assistente como a equipe anterior o construiu. Chunk de tamanho fixo sobre a central inteira, recuperação por similaridade e geração com os trechos no prompt. O prompt traz o papel do assistente, o escopo e a instrução de responder só com o que está nos trechos.

A medição da recuperação no conjunto de referência é a fração das 62 perguntas em que `resposta_referencia` aparece no texto dos trechos recuperados, com `data_referencia` como data de referência da vigência. A linha de base ignora a vigência, porque a equipe anterior a ignorava.

In [ ]:
# ========================== SUA VEZ ==========================
# Indexação da linha de base. Chunk de tamanho fixo sobre todos os documentos, embedding e índice.

In [ ]:
# ========================== SUA VEZ ==========================
# Recuperação e geração. O prompt do assistente, com papel, escopo e a instrução de responder só com os trechos.

In [ ]:
# Medição da recuperação no conjunto de referência. A variável `recupera(pergunta, data)` devolve a lista de trechos.
# Esta célula fica como está. Ela é a mesma para todas as configurações.
acertos = 0
erradas = []
for p in perguntas:
    trechos = recupera(p["pergunta"], p["data_referencia"])
    hit = any(p["resposta_referencia"].lower() in t.lower() for t in trechos)
    acertos += hit
    if not hit:
        erradas.append(p["id"])
print(f"linha de base. {acertos} de {len(perguntas)}")
print("erradas.", erradas)

**Análise da linha de base.** [Leitura do número. Que perguntas erram, e o que elas têm em comum. A decisão, o que o grupo vai mudar primeiro e por quê.]

## 2. Melhorias, uma de cada vez

Cada configuração nova muda uma coisa em relação à anterior. A célula de medição é a mesma da seção 1. Repita o bloco abaixo (mudança, medição, análise) para cada configuração testada. Curadoria, chunking por tipo com metadados, filtro de vigência pela data, busca híbrida, re-ranking, parent-document retrieval e a classe do chamado como metadado são as técnicas da disciplina. A ordem e a escolha são do grupo.

### Configuração 1. [nome da mudança]

In [ ]:
# ========================== SUA VEZ ==========================
# O que muda em relação à configuração anterior, e só isso.

In [ ]:
# Medição, a mesma da seção 1.
acertos = 0
erradas = []
for p in perguntas:
    trechos = recupera(p["pergunta"], p["data_referencia"])
    hit = any(p["resposta_referencia"].lower() in t.lower() for t in trechos)
    acertos += hit
    if not hit:
        erradas.append(p["id"])
print(f"configuração 1. {acertos} de {len(perguntas)}")
print("erradas.", erradas)

**Análise da configuração 1.** [O número contra o anterior. Que perguntas ganhou e que perguntas perdeu. A decisão.]

### Configuração 2. [nome da mudança]

In [ ]:
# ========================== SUA VEZ ==========================

In [ ]:
# Medição, a mesma da seção 1.

**Análise da configuração 2.** [...]

## 3. Avaliação com RAGAS

Para as configurações que o grupo considera candidatas. Fidelidade, precisão do contexto e relevância da resposta, sobre uma amostra do conjunto de referência, com o mesmo modelo e temperatura zero.

In [ ]:
# ========================== SUA VEZ ==========================

**Análise do RAGAS.** [O que cada métrica diz sobre cada configuração candidata. Qual configuração segue para a rodada dos chamados, e por quê.]

## 4. Rodada nos 200 chamados

Para a configuração escolhida. O assistente responde cada chamado, com o texto da reclamação como consulta e a data de abertura como data de referência. A resposta é julgada por um modelo em quatro critérios. O prompt do juiz abaixo é o ponto de partida, e o grupo pode ajustá-lo desde que os quatro critérios continuem os mesmos.

In [ ]:
PROMPT_JUIZ = """Você avalia a resposta de um atendente a um chamado de cliente. Responda só com JSON.

Chamado. {texto}
Resposta do atendente. {resposta}
Trechos dos documentos que o atendente recebeu. {docs}

Avalie quatro coisas.
desfecho. "sim" se a resposta diz ao cliente o que vai acontecer com o chamado, com a política aplicável, "nao" caso contrário.
parametro. "sim" se a resposta informa o prazo ou o valor aplicável ao caso, "nao" se informa outro ou omite, "na" se o caso não tem prazo nem valor.
inventou. "sim" se a resposta afirma prazo, valor ou regra que não está nos trechos recebidos, "nao" caso contrário.
abstencao. "sim" se a resposta declara que os documentos não cobrem o caso ou que falta informação, "nao" caso contrário.

JSON no formato {{"desfecho": "...", "parametro": "...", "inventou": "...", "abstencao": "..."}}"""

In [ ]:
# ========================== SUA VEZ ==========================
# Geração da resposta de cada chamado com a configuração escolhida, julgamento e contagem dos quatro critérios.

**Análise da rodada.** [Os quatro números. Em que classes o assistente erra o desfecho. Onde inventa. Se a abstenção aparece onde deveria. O que o grupo mudaria na base a partir disso.]

## 5. Conclusão

Responda com os números do próprio trabalho.

1. Qual configuração venceu, e por quê.
2. Onde o assistente ainda erra, e em que etapa, recuperação ou geração.
3. Se algum resultado ficou fraco, qual é a causa investigada.
4. O que o grupo faria em seguida.